In [ ]:
# Cell 1 — Setup & Imports
import os, torch
from src.render import render_pdf
from src.embed import get_device, load_model, embed_tiles, embed_query
from src.index import build_index, load_index, search
from src.reader import answer

device = get_device()
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 2 — Render PDF → Tiles
import matplotlib.pyplot as plt
from PIL import Image

PDF_PATH = 'pixelrag-paper.pdf'
tiles = render_pdf(PDF_PATH, 'data/tiles')
print(f'Rendered {len(tiles)} tiles')

cols = min(6, len(tiles))
rows = (len(tiles) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 4))
flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]
for i, (tile, ax) in enumerate(zip(tiles, flat)):
    ax.imshow(Image.open(tile['path']))
    ax.set_title(f"Page {tile['page']}", fontsize=9)
    ax.axis('off')
for ax in flat[len(tiles):]:
    ax.axis('off')
plt.suptitle('Rendered PDF Tiles', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 3 — Load Embedding Model + LoRA
print('Loading Qwen3-VL-Embedding-2B + LoRA adapters...')
print('This may take several minutes on first run.\n')
model, processor, device, lora_loaded = load_model(device)
print(f'Model type  : {type(model).__name__}')
print(f'Device      : {device}')
print(f'LoRA loaded : {lora_loaded}')

In [ ]:
# Cell 4 — Embed Tiles → Vectors + t-SNE
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

embeddings, metadata = embed_tiles(tiles, model, processor, device, 'data/embeddings')
print(f'Embedding shape : {embeddings.shape}')
print(f'dtype           : {embeddings.dtype}')
print(f'Sample L2 norm  : {np.linalg.norm(embeddings[0]):.4f}  (should be ~1.0)')

if len(embeddings) >= 3:
    perplexity = min(5, len(embeddings) - 1)
    coords = TSNE(n_components=2, perplexity=perplexity, random_state=42).fit_transform(embeddings)
    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(coords[:, 0], coords[:, 1],
                    c=range(len(coords)), cmap='tab20', s=140, zorder=5)
    for i, (x, y) in enumerate(coords):
        ax.annotate(f"p{metadata[i]['page']}", (x, y),
                    fontsize=8, ha='center', va='bottom',
                    xytext=(0, 6), textcoords='offset points')
    plt.colorbar(sc, ax=ax, label='Page index')
    ax.set_title('t-SNE of tile embeddings (coloured by page)', fontsize=12)
    ax.set_xlabel('t-SNE dim 1')
    ax.set_ylabel('t-SNE dim 2')
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 5 — Build FAISS Index
index, index_meta = build_index(embeddings, metadata, 'data/index')
print(f'Index type  : {type(index).__name__}')
print(f'Vectors     : {index.ntotal}')
print(f'Dimension   : {index.d}')

In [ ]:
# Cell 6 — Query Demo (3 example questions)
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

EXAMPLE_QUERIES = [
    'What embedding model does PixelRAG use?',
    'How does PixelRAG accuracy compare to text-based RAG?',
    'Describe the FAISS indexing strategy in PixelRAG.',
]
api_key = os.environ.get('ANTHROPIC_API_KEY')

for query in EXAMPLE_QUERIES:
    print(f'\n{"="*65}')
    print(f'Q: {query}')
    print('='*65)

    q_vec = embed_query(query, model, processor, device)
    results = search(q_vec, index, index_meta, k=3)

    fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 6))
    if len(results) == 1:
        axes = [axes]
    for ax, tile in zip(axes, results):
        ax.imshow(mpimg.imread(tile['path']))
        ax.set_title(f"Page {tile['page']}  score {tile['score']:.2f}", fontsize=9)
        ax.axis('off')
    plt.suptitle(f'Retrieved tiles for: {query[:55]}...', fontsize=10)
    plt.tight_layout()
    plt.show()

    if api_key:
        ans, _ = answer(query, results, [], api_key=api_key)
        print(f'A: {ans}')
    else:
        print('A: [Set ANTHROPIC_API_KEY to enable answering]')

In [ ]:
# Cell 7 — Interactive Query (change query and re-run this cell)
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

query = 'What is the overall architecture of PixelRAG?'

q_vec = embed_query(query, model, processor, device)
results = search(q_vec, index, index_meta, k=3)

fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 6))
if len(results) == 1:
    axes = [axes]
for ax, tile in zip(axes, results):
    ax.imshow(mpimg.imread(tile['path']))
    ax.set_title(f"Page {tile['page']}  score {tile['score']:.2f}", fontsize=9)
    ax.axis('off')
plt.suptitle(f'Query: {query}', fontsize=11)
plt.tight_layout()
plt.show()

api_key = os.environ.get('ANTHROPIC_API_KEY')
if api_key:
    ans, _ = answer(query, results, [], api_key=api_key)
    print(f'A: {ans}')
else:
    print('[Set ANTHROPIC_API_KEY to enable answering]')